# MLB 2025 Season Dataset Builder
## Schedule + Attendance + Uniforms

This notebook fetches and combines data from three MLB APIs:
1. Schedule API - game schedule and results
2. Attendance API - game attendance data
3. Uniforms API - team uniform information per game


In [2]:
import requests
import pandas as pd
import json
from datetime import datetime
import time
from tqdm import tqdm


## Step 1: Fetch Schedule Data


In [3]:
# Fetch the complete 2025 schedule
schedule_url = "https://statsapi.mlb.com/api/v1/schedule?gameType=R&season=2025&sportId=1"
print("Fetching schedule data...")
response = requests.get(schedule_url)
schedule_data = response.json()

print(f"Total games: {schedule_data['totalGames']}")
print(f"Total dates: {len(schedule_data['dates'])}")


Fetching schedule data...
Total games: 2464
Total dates: 184


## Step 2: Parse Schedule and Extract Game Information


In [4]:
# Parse schedule data into a flat structure
games_list = []

for date_entry in schedule_data['dates']:
    date = date_entry['date']
    
    for game in date_entry['games']:
        game_info = {
            'gamePk': game['gamePk'],
            'date': date,
            'gameDate': game['gameDate'],
            'season': game['season'],
            'gameType': game['gameType'],
            'status': game['status']['detailedState'],
            'statusCode': game['status']['codedGameState'],
            
            # Home team info
            'home_team_id': game['teams']['home']['team']['id'],
            'home_team_name': game['teams']['home']['team']['name'],
            'home_score': game['teams']['home'].get('score'),
            'home_wins': game['teams']['home']['leagueRecord'].get('wins'),
            'home_losses': game['teams']['home']['leagueRecord'].get('losses'),
            'home_pct': game['teams']['home']['leagueRecord'].get('pct'),
            'home_is_winner': game['teams']['home'].get('isWinner'),
            
            # Away team info
            'away_team_id': game['teams']['away']['team']['id'],
            'away_team_name': game['teams']['away']['team']['name'],
            'away_score': game['teams']['away'].get('score'),
            'away_wins': game['teams']['away']['leagueRecord'].get('wins'),
            'away_losses': game['teams']['away']['leagueRecord'].get('losses'),
            'away_pct': game['teams']['away']['leagueRecord'].get('pct'),
            'away_is_winner': game['teams']['away'].get('isWinner'),
            
            # Venue info
            'venue_id': game['venue']['id'],
            'venue_name': game['venue']['name'],
            
            # Game details
            'doubleHeader': game['doubleHeader'],
            'gameNumber': game['gameNumber'],
            'dayNight': game.get('dayNight'),
            'description': game.get('description', ''),
            'scheduledInnings': game['scheduledInnings'],
            'seriesGameNumber': game['seriesGameNumber'],
            'gamesInSeries': game['gamesInSeries']
        }
        games_list.append(game_info)

# Create DataFrame
df_games = pd.DataFrame(games_list)
print(f"\nTotal games parsed: {len(df_games)}")
print(f"\nFirst few games:")
df_games.head()



Total games parsed: 2464

First few games:


,gamePk,date,gameDate,season,gameType,status,statusCode,home_team_id,home_team_name,home_score,...,away_is_winner,venue_id,venue_name,doubleHeader,gameNumber,dayNight,description,scheduledInnings,seriesGameNumber,gamesInSeries
0,778563,2025-03-18,2025-03-18T10:10:00Z,2025,R,Final,F,112,Chicago Cubs,1.0,...,True,2397,Tokyo Dome,N,1,night,"in Tokyo, Japan",9,1,2
1,778564,2025-03-19,2025-03-19T10:10:00Z,2025,R,Final,F,112,Chicago Cubs,3.0,...,True,2397,Tokyo Dome,N,1,night,"in Tokyo, Japan",9,2,2
2,778557,2025-03-27,2025-03-27T19:05:00Z,2025,R,Final,F,147,New York Yankees,4.0,...,False,3313,Yankee Stadium,N,1,day,Yankees home opener,9,1,3
3,778556,2025-03-27,2025-03-27T19:07:00Z,2025,R,Final,F,141,Toronto Blue Jays,2.0,...,True,14,Rogers Centre,N,1,day,Blue Jays home opener,9,1,4
4,778553,2025-03-27,2025-03-27T20:05:00Z,2025,R,Final,F,140,Texas Rangers,2.0,...,True,5325,Globe Life Field,N,1,day,Rangers home opener,9,1,4


In [7]:
# Check for and remove duplicate gamePk (postponed/rescheduled games)
print("Checking for duplicate games...")
duplicates_before = len(df_games)
duplicate_games = df_games[df_games.duplicated(subset=['gamePk'], keep=False)]

if len(duplicate_games) > 0:
    print(f"\nFound {duplicate_games['gamePk'].nunique()} games with multiple entries (postponed/rescheduled)")
    print(f"Total duplicate rows: {len(duplicate_games)}")
    
    # Show examples of duplicates
    print("\nExample duplicates (showing first game with multiple dates):")
    first_dup_gamepk = duplicate_games['gamePk'].iloc[0]
    print(df_games[df_games['gamePk'] == first_dup_gamepk][['gamePk', 'date', 'home_team_name', 'away_team_name', 'status']])
    
    # Keep only the game with the latest date for each gamePk
    df_games = df_games.sort_values('date', ascending=False).drop_duplicates(subset=['gamePk'], keep='first')
    df_games = df_games.sort_values('date').reset_index(drop=True)
    
    duplicates_after = len(df_games)
    print(f"\nRemoved {duplicates_before - duplicates_after} duplicate entries")
    print(f"Final game count: {duplicates_after}")
else:
    print("No duplicate gamePk found!")
    
print(f"\nVerifying: Unique gamePk count = {df_games['gamePk'].nunique()} (should equal {len(df_games)})")


Checking for duplicate games...

Found 34 games with multiple entries (postponed/rescheduled)
Total duplicate rows: 68

Example duplicates (showing first game with multiple dates):
     gamePk        date  home_team_name       away_team_name     status
128  778443  2025-04-05  Boston Red Sox  St. Louis Cardinals  Postponed
130  778443  2025-04-06  Boston Red Sox  St. Louis Cardinals      Final

Removed 34 duplicate entries
Final game count: 2430

Verifying: Unique gamePk count = 2430 (should equal 2430)


## Step 3: Fetch Attendance Data for Each Game


In [9]:
# Function to fetch attendance data
def fetch_attendance(date, team_id):
    """
    Fetch attendance data for a specific date and team.
    Returns a dictionary with attendance information including gamePk matching.
    """
    url = f"https://statsapi.mlb.com/api/v1/attendance?sportId=1&date={date}&gameType=R&teamId={team_id}"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            
            # The API returns attendance records
            if 'records' in data and len(data['records']) > 0:
                record = data['records'][0]
                
                # Extract attendance info with gamePk references
                result = {
                    'attendanceHigh': record.get('attendanceHigh'),
                    'attendanceLow': record.get('attendanceLow'),
                    'attendanceHighGamePk': record.get('attendanceHighGame', {}).get('gamePk') if record.get('attendanceHighGame') else None,
                    'attendanceLowGamePk': record.get('attendanceLowGame', {}).get('gamePk') if record.get('attendanceLowGame') else None,
                    'gamesHomeTotal': record.get('gamesHomeTotal', 1)
                }
                return result
        return None
    except Exception as e:
        print(f"Error fetching attendance for date={date}, team={team_id}: {e}")
        return None


In [10]:
# Fetch attendance data for each game
print("Fetching attendance data...")
attendance_data = []

# Process games in batches to show progress
for idx, row in tqdm(df_games.iterrows(), total=len(df_games), desc="Fetching attendance"):
    attendance_info = fetch_attendance(row['date'], row['home_team_id'])
    
    if attendance_info:
        # Match by gamePk to determine which attendance applies
        current_gamePk = row['gamePk']
        
        # Check if this game is the high or low attendance game
        if attendance_info.get('gamesHomeTotal') > 1:
            # Double header - match by gamePk
            if current_gamePk == attendance_info.get('attendanceHighGamePk'):
                actual_attendance = attendance_info['attendanceHigh']
            elif current_gamePk == attendance_info.get('attendanceLowGamePk'):
                actual_attendance = attendance_info['attendanceLow']
            else:
                actual_attendance = None  # Shouldn't happen
        else:
            # Single game - attendanceHigh and attendanceLow are the same
            actual_attendance = attendance_info['attendanceHigh']
        
        attendance_data.append({
            'gamePk': row['gamePk'],
            'attendance': actual_attendance,
            'attendance_high_for_date': attendance_info.get('attendanceHigh'),
            'attendance_low_for_date': attendance_info.get('attendanceLow'),
            'is_doubleheader_date': attendance_info.get('gamesHomeTotal', 1) > 1
        })
    else:
        attendance_data.append({
            'gamePk': row['gamePk'],
            'attendance': None,
            'attendance_high_for_date': None,
            'attendance_low_for_date': None,
            'is_doubleheader_date': False
        })
    
    # Small delay to avoid overwhelming the API
    if idx % 50 == 0:
        time.sleep(0.5)

df_attendance = pd.DataFrame(attendance_data)
print(f"\nAttendance data fetched for {len(df_attendance)} games")
print(f"Games with attendance data: {df_attendance['attendance'].notna().sum()}")


Fetching attendance data...


Fetching attendance:  19%|█▉        | 472/2430 [03:32<5:25:57,  9.99s/it]

Error fetching attendance for date=2025-05-02, team=134: HTTPSConnectionPool(host='statsapi.mlb.com', port=443): Read timed out. (read timeout=None)


Fetching attendance:  19%|█▉        | 473/2430 [04:02<8:41:42, 16.00s/it]

Error fetching attendance for date=2025-05-02, team=110: HTTPSConnectionPool(host='statsapi.mlb.com', port=443): Max retries exceeded with url: /api/v1/attendance?sportId=1&date=2025-05-02&gameType=R&teamId=110 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x16c996ab0>: Failed to resolve 'statsapi.mlb.com' ([Errno 8] nodename nor servname provided, or not known)"))


Fetching attendance:  20%|█▉        | 474/2430 [13:47<101:21:27, 186.55s/it]

Error fetching attendance for date=2025-05-02, team=147: HTTPSConnectionPool(host='statsapi.mlb.com', port=443): Max retries exceeded with url: /api/v1/attendance?sportId=1&date=2025-05-02&gameType=R&teamId=147 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x16c9971d0>: Failed to resolve 'statsapi.mlb.com' ([Errno 8] nodename nor servname provided, or not known)"))


Fetching attendance: 100%|██████████| 2430/2430 [36:24<00:00,  1.11it/s]    


Attendance data fetched for 2430 games
Games with attendance data: 2423


## Step 3b: Retry Missing Attendance Data


In [18]:
# Retry fetching attendance for games with missing data
missing_attendance = df_attendance[df_attendance['attendance'].isna()]

if len(missing_attendance) > 0:
    print(f"\nRetrying attendance fetch for {len(missing_attendance)} games with missing data...")
    
    # Get the corresponding game info from df_games
    missing_gamepks = missing_attendance['gamePk'].tolist()
    games_to_retry = df_games[df_games['gamePk'].isin(missing_gamepks)]
    
    retry_count = 0
    for idx, row in tqdm(games_to_retry.iterrows(), total=len(games_to_retry), desc="Retrying"):
        attendance_info = fetch_attendance(row['date'], row['home_team_id'])
        
        if attendance_info:
            current_gamePk = row['gamePk']
            
            # Check if this game is the high or low attendance game
            if attendance_info.get('gamesHomeTotal') > 1:
                # Double header - match by gamePk
                if current_gamePk == attendance_info.get('attendanceHighGamePk'):
                    actual_attendance = attendance_info['attendanceHigh']
                elif current_gamePk == attendance_info.get('attendanceLowGamePk'):
                    actual_attendance = attendance_info['attendanceLow']
                else:
                    actual_attendance = None
            else:
                # Single game
                actual_attendance = attendance_info['attendanceHigh']
            
            # Update the dataframe if we got attendance
            if actual_attendance is not None:
                df_attendance.loc[df_attendance['gamePk'] == current_gamePk, 'attendance'] = actual_attendance
                df_attendance.loc[df_attendance['gamePk'] == current_gamePk, 'attendance_high_for_date'] = attendance_info.get('attendanceHigh')
                df_attendance.loc[df_attendance['gamePk'] == current_gamePk, 'attendance_low_for_date'] = attendance_info.get('attendanceLow')
                df_attendance.loc[df_attendance['gamePk'] == current_gamePk, 'is_doubleheader_date'] = attendance_info.get('gamesHomeTotal', 1) > 1
                retry_count += 1
        
        time.sleep(1)  # Longer delay for retries
    
    print(f"\nSuccessfully retrieved {retry_count} additional attendance records")
    print(f"Remaining missing: {df_attendance['attendance'].isna().sum()}")
else:
    print("\nNo missing attendance data to retry!")



Retrying attendance fetch for 4 games with missing data...


Retrying: 100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


Successfully retrieved 0 additional attendance records
Remaining missing: 4


In [24]:
# Show which games still have missing attendance after retry
still_missing = df_attendance[df_attendance['attendance'].isna()]

if len(still_missing) > 0:
    print(f"\nGames still missing attendance: {len(still_missing)}")
    
    # Merge with game info to see details
    missing_with_info = still_missing.merge(
        df_games[['gamePk', 'date', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'status', 'venue_name']], 
        on='gamePk', 
        how='left'
    )
    
    print("\nDetails of games with missing attendance:")
    display(missing_with_info[['gamePk', 'date', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'status', 'venue_name']])
else:
    print("\n✅ All games have attendance data!")



Games still missing attendance: 1

Details of games with missing attendance:


,gamePk,date,home_team_id,home_team_name,away_team_id,away_team_name,status,venue_name
0,776907,2025-08-03,113,Cincinnati Reds,144,Atlanta Braves,Final,Bristol Motor Speedway


In [22]:
# For double headers with identical attendance (where both high/low have same gamePk),
# fill the missing game with the attendance from the matched game
still_missing = df_attendance[df_attendance['attendance'].isna()]

if len(still_missing) > 0:
    print(f"\nHandling {len(still_missing)} double header games with identical attendance...")
    
    filled_count = 0
    for idx, missing_row in still_missing.iterrows():
        missing_gamepk = missing_row['gamePk']
        
        # Get the game info
        game_info = df_games[df_games['gamePk'] == missing_gamepk].iloc[0]
        date = game_info['date']
        home_team_id = game_info['home_team_id']
        
        # Find other games on the same date for the same home team
        same_date_games = df_games[
            (df_games['date'] == date) & 
            (df_games['home_team_id'] == home_team_id) &
            (df_games['gamePk'] != missing_gamepk)
        ]
        
        if len(same_date_games) > 0:
            # Check if the other game has attendance
            other_gamepk = same_date_games.iloc[0]['gamePk']
            other_attendance = df_attendance[df_attendance['gamePk'] == other_gamepk]['attendance'].iloc[0]
            
            if pd.notna(other_attendance):
                # Use the same attendance for this game
                df_attendance.loc[df_attendance['gamePk'] == missing_gamepk, 'attendance'] = other_attendance
                
                # Also copy the high/low values
                other_row = df_attendance[df_attendance['gamePk'] == other_gamepk].iloc[0]
                df_attendance.loc[df_attendance['gamePk'] == missing_gamepk, 'attendance_high_for_date'] = other_row['attendance_high_for_date']
                df_attendance.loc[df_attendance['gamePk'] == missing_gamepk, 'attendance_low_for_date'] = other_row['attendance_low_for_date']
                df_attendance.loc[df_attendance['gamePk'] == missing_gamepk, 'is_doubleheader_date'] = True
                
                filled_count += 1
                print(f"  - Filled gamePk {missing_gamepk} with attendance {other_attendance:,.0f} from gamePk {other_gamepk}")
    
    print(f"\nFilled {filled_count} games using attendance from their double header partner")
    print(f"Final missing count: {df_attendance['attendance'].isna().sum()}")
else:
    print("\nNo missing attendance to fill!")



Handling 4 double header games with identical attendance...
  - Filled gamePk 777861 with attendance 26,142 from gamePk 777829
  - Filled gamePk 777623 with attendance 27,951 from gamePk 777612
  - Filled gamePk 777294 with attendance 35,845 from gamePk 777277

Filled 3 games using attendance from their double header partner
Final missing count: 1


In [ ]:
# Manual override for known edge cases (postponed games with date mismatch)
# Game 776907: Postponed from Aug 2 → Aug 3, but attendance API kept Aug 2 date
# Bristol Motor Speedway
# https://statsapi.mlb.com/api/v1/attendance?sportId=1&date=2025-08-02&gameType=R&teamId=113
manual_overrides = {
    776907: {
        'attendance': 91032,
        'attendance_high_for_date': 91032,
        'attendance_low_for_date': 91032,
        'is_doubleheader_date': False,
        'note': 'Postponed from 2025-08-02 to 2025-08-03, attendance API date mismatch'
    }
}

if len(manual_overrides) > 0:
    print(f"\nApplying {len(manual_overrides)} manual attendance override(s)...")
    
    for gamepk, override_data in manual_overrides.items():
        if gamepk in df_attendance['gamePk'].values:
            df_attendance.loc[df_attendance['gamePk'] == gamepk, 'attendance'] = override_data['attendance']
            df_attendance.loc[df_attendance['gamePk'] == gamepk, 'attendance_high_for_date'] = override_data['attendance_high_for_date']
            df_attendance.loc[df_attendance['gamePk'] == gamepk, 'attendance_low_for_date'] = override_data['attendance_low_for_date']
            df_attendance.loc[df_attendance['gamePk'] == gamepk, 'is_doubleheader_date'] = override_data['is_doubleheader_date']
            
            print(f"  ✓ gamePk {gamepk}: {override_data['attendance']:,} - {override_data['note']}")
        else:
            print(f"  ⚠️ gamePk {gamepk} not found in df_attendance")
    
    print(f"\nOverrides applied. Missing count: {df_attendance['attendance'].isna().sum()}")



Applying 1 manual attendance override(s)...
  ✓ gamePk 776907: 91,032 - Postponed from 2025-08-02 to 2025-08-03, attendance API date mismatch

Overrides applied. Missing count: 0


In [28]:
# Final summary of attendance data completeness
print("=" * 50)
print("ATTENDANCE DATA SUMMARY")
print("=" * 50)
print(f"\nTotal games: {len(df_attendance)}")
print(f"Games with attendance: {df_attendance['attendance'].notna().sum()}")
print(f"Games missing attendance: {df_attendance['attendance'].isna().sum()}")
print(f"Completion rate: {df_attendance['attendance'].notna().sum() / len(df_attendance) * 100:.2f}%")

final_missing = df_attendance[df_attendance['attendance'].isna()]
if len(final_missing) > 0:
    print(f"\n⚠️ {len(final_missing)} games still missing attendance - likely postponed/cancelled games")
else:
    print("\n✅ All games have attendance data!")


ATTENDANCE DATA SUMMARY

Total games: 2430
Games with attendance: 2430
Games missing attendance: 0
Completion rate: 100.00%

✅ All games have attendance data!


In [30]:
df_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2430 entries, 0 to 2429
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   gamePk                    2430 non-null   int64  
 1   attendance                2430 non-null   float64
 2   attendance_high_for_date  2430 non-null   float64
 3   attendance_low_for_date   2430 non-null   float64
 4   is_doubleheader_date      2430 non-null   bool   
dtypes: bool(1), float64(3), int64(1)
memory usage: 78.4 KB


## Step 4: Fetch Uniform Data for Each Game


In [31]:
# Function to fetch uniform data
def fetch_uniforms(game_pk):
    """
    Fetch uniform data for a specific game.
    Returns a dictionary with uniform information for home and away teams.
    """
    url = f"https://statsapi.mlb.com/api/v1/uniforms/game?gamePks={game_pk}"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            
            result = {
                'home_jersey': None,
                'home_pants': None,
                'home_cap': None,
                'home_jersey_code': None,
                'away_jersey': None,
                'away_pants': None,
                'away_cap': None,
                'away_jersey_code': None
            }
            
            # Parse uniform data
            if 'uniforms' in data and len(data['uniforms']) > 0:
                game_uniform = data['uniforms'][0]
                
                # Parse home uniforms
                if 'home' in game_uniform and 'uniformAssets' in game_uniform['home']:
                    for asset in game_uniform['home']['uniformAssets']:
                        asset_type = asset.get('uniformAssetType', {}).get('uniformAssetTypeText', '')
                        if asset_type == 'Jersey':
                            result['home_jersey'] = asset.get('uniformAssetText')
                            result['home_jersey_code'] = asset.get('uniformAssetCode')
                        elif asset_type == 'Pants':
                            result['home_pants'] = asset.get('uniformAssetText')
                        elif asset_type == 'Cap':
                            result['home_cap'] = asset.get('uniformAssetText')
                
                # Parse away uniforms
                if 'away' in game_uniform and 'uniformAssets' in game_uniform['away']:
                    for asset in game_uniform['away']['uniformAssets']:
                        asset_type = asset.get('uniformAssetType', {}).get('uniformAssetTypeText', '')
                        if asset_type == 'Jersey':
                            result['away_jersey'] = asset.get('uniformAssetText')
                            result['away_jersey_code'] = asset.get('uniformAssetCode')
                        elif asset_type == 'Pants':
                            result['away_pants'] = asset.get('uniformAssetText')
                        elif asset_type == 'Cap':
                            result['away_cap'] = asset.get('uniformAssetText')
            
            return result
        return None
    except Exception as e:
        print(f"Error fetching uniforms for gamePk={game_pk}: {e}")
        return None


In [32]:
# Fetch uniform data for each game
print("Fetching uniform data...")
uniform_data = []

for idx, game_pk in tqdm(enumerate(df_games['gamePk']), total=len(df_games), desc="Fetching uniforms"):
    uniform_info = fetch_uniforms(game_pk)
    
    if uniform_info:
        uniform_data.append({
            'gamePk': game_pk,
            'home_jersey': uniform_info['home_jersey'],
            'home_pants': uniform_info['home_pants'],
            'home_cap': uniform_info['home_cap'],
            'home_jersey_code': uniform_info['home_jersey_code'],
            'away_jersey': uniform_info['away_jersey'],
            'away_pants': uniform_info['away_pants'],
            'away_cap': uniform_info['away_cap'],
            'away_jersey_code': uniform_info['away_jersey_code']
        })
    else:
        uniform_data.append({
            'gamePk': game_pk,
            'home_jersey': None,
            'home_pants': None,
            'home_cap': None,
            'home_jersey_code': None,
            'away_jersey': None,
            'away_pants': None,
            'away_cap': None,
            'away_jersey_code': None
        })
    
    # Small delay to avoid overwhelming the API
    if idx % 50 == 0:
        time.sleep(0.5)

df_uniforms = pd.DataFrame(uniform_data)
print(f"\nUniform data fetched for {len(df_uniforms)} games")
print(f"Games with home jersey data: {df_uniforms['home_jersey'].notna().sum()}")
print(f"Games with away jersey data: {df_uniforms['away_jersey'].notna().sum()}")


Fetching uniform data...


Fetching uniforms: 100%|██████████| 2430/2430 [08:12<00:00,  4.94it/s]


Uniform data fetched for 2430 games
Games with home jersey data: 2428
Games with away jersey data: 2429


## Step 4b: Retry Missing Uniform Data


In [33]:
# Retry fetching uniforms for games with missing data
missing_uniforms = df_uniforms[df_uniforms['home_jersey'].isna() | df_uniforms['away_jersey'].isna()]

if len(missing_uniforms) > 0:
    print(f"\nRetrying uniform fetch for {len(missing_uniforms)} games with missing data...")
    
    missing_gamepks = missing_uniforms['gamePk'].tolist()
    
    retry_count = 0
    for game_pk in tqdm(missing_gamepks, desc="Retrying uniforms"):
        uniform_info = fetch_uniforms(game_pk)
        
        if uniform_info and (uniform_info.get('home_jersey') or uniform_info.get('away_jersey')):
            # Update the dataframe if we got uniform data
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'home_jersey'] = uniform_info['home_jersey']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'home_pants'] = uniform_info['home_pants']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'home_cap'] = uniform_info['home_cap']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'home_jersey_code'] = uniform_info['home_jersey_code']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'away_jersey'] = uniform_info['away_jersey']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'away_pants'] = uniform_info['away_pants']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'away_cap'] = uniform_info['away_cap']
            df_uniforms.loc[df_uniforms['gamePk'] == game_pk, 'away_jersey_code'] = uniform_info['away_jersey_code']
            retry_count += 1
        
        time.sleep(1)  # Longer delay for retries
    
    print(f"\nSuccessfully retrieved {retry_count} additional uniform records")
    print(f"Remaining missing home jerseys: {df_uniforms['home_jersey'].isna().sum()}")
    print(f"Remaining missing away jerseys: {df_uniforms['away_jersey'].isna().sum()}")
else:
    print("\nNo missing uniform data to retry!")



Retrying uniform fetch for 2 games with missing data...


Retrying uniforms: 100%|██████████| 2/2 [00:02<00:00,  1.21s/it]


Successfully retrieved 1 additional uniform records
Remaining missing home jerseys: 2
Remaining missing away jerseys: 1


In [34]:
# Show which games still have missing uniforms after retry
still_missing_uniforms = df_uniforms[df_uniforms['home_jersey'].isna() | df_uniforms['away_jersey'].isna()]

if len(still_missing_uniforms) > 0:
    print(f"\nGames still missing uniform data: {len(still_missing_uniforms)}")
    
    # Merge with game info to see details
    missing_with_info = still_missing_uniforms.merge(
        df_games[['gamePk', 'date', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'status', 'venue_name']], 
        on='gamePk', 
        how='left'
    )
    
    print("\nDetails of games with missing uniforms:")
    display(missing_with_info[['gamePk', 'date', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'status', 'venue_name', 'home_jersey', 'away_jersey']])
else:
    print("\n✅ All games have uniform data!")



Games still missing uniform data: 2

Details of games with missing uniforms:


,gamePk,date,home_team_id,home_team_name,away_team_id,away_team_name,status,venue_name,home_jersey,away_jersey
0,776823,2025-08-08,119,Los Angeles Dodgers,141,Toronto Blue Jays,Final,Dodger Stadium,None,None
1,776714,2025-08-17,111,Boston Red Sox,146,Miami Marlins,Final,Fenway Park,None,Marlins Road Grey Jersey


In [36]:
# Fill missing uniforms with most common jersey for that team
# Merge uniforms with game info to get team IDs
df_uniforms_with_teams = df_uniforms.merge(
    df_games[['gamePk', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name']], 
    on='gamePk', 
    how='left'
)

# Fill missing home jerseys
missing_home = df_uniforms_with_teams[df_uniforms_with_teams['home_jersey'].isna()]
if len(missing_home) > 0:
    print(f"\nFilling {len(missing_home)} missing home jerseys with most common jersey for each team...")
    
    for idx, row in missing_home.iterrows():
        team_id = row['home_team_id']
        team_name = row['home_team_name']
        
        # Find most common home jersey for this team
        team_home_jerseys = df_uniforms_with_teams[
            (df_uniforms_with_teams['home_team_id'] == team_id) & 
            (df_uniforms_with_teams['home_jersey'].notna())
        ]['home_jersey']
        
        if len(team_home_jerseys) > 0:
            most_common_jersey = team_home_jerseys.mode()[0]
            
            # Also get the most common pants and cap for this jersey
            team_outfit = df_uniforms_with_teams[
                (df_uniforms_with_teams['home_team_id'] == team_id) & 
                (df_uniforms_with_teams['home_jersey'] == most_common_jersey)
            ]
            most_common_pants = team_outfit['home_pants'].mode()[0] if len(team_outfit['home_pants'].dropna()) > 0 else None
            most_common_cap = team_outfit['home_cap'].mode()[0] if len(team_outfit['home_cap'].dropna()) > 0 else None
            most_common_code = team_outfit['home_jersey_code'].mode()[0] if len(team_outfit['home_jersey_code'].dropna()) > 0 else None
            
            # Update in df_uniforms
            gamepk = row['gamePk']
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'home_jersey'] = most_common_jersey
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'home_pants'] = most_common_pants
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'home_cap'] = most_common_cap
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'home_jersey_code'] = most_common_code
            
            print(f"  ✓ gamePk {gamepk} ({team_name}): {most_common_jersey}")

# Fill missing away jerseys
missing_away = df_uniforms_with_teams[df_uniforms_with_teams['away_jersey'].isna()]
if len(missing_away) > 0:
    print(f"\nFilling {len(missing_away)} missing away jerseys with most common jersey for each team...")
    
    for idx, row in missing_away.iterrows():
        team_id = row['away_team_id']
        team_name = row['away_team_name']
        
        # Find most common away jersey for this team
        team_away_jerseys = df_uniforms_with_teams[
            (df_uniforms_with_teams['away_team_id'] == team_id) & 
            (df_uniforms_with_teams['away_jersey'].notna())
        ]['away_jersey']
        
        if len(team_away_jerseys) > 0:
            most_common_jersey = team_away_jerseys.mode()[0]
            
            # Also get the most common pants and cap for this jersey
            team_outfit = df_uniforms_with_teams[
                (df_uniforms_with_teams['away_team_id'] == team_id) & 
                (df_uniforms_with_teams['away_jersey'] == most_common_jersey)
            ]
            most_common_pants = team_outfit['away_pants'].mode()[0] if len(team_outfit['away_pants'].dropna()) > 0 else None
            most_common_cap = team_outfit['away_cap'].mode()[0] if len(team_outfit['away_cap'].dropna()) > 0 else None
            most_common_code = team_outfit['away_jersey_code'].mode()[0] if len(team_outfit['away_jersey_code'].dropna()) > 0 else None
            
            # Update in df_uniforms
            gamepk = row['gamePk']
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'away_jersey'] = most_common_jersey
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'away_pants'] = most_common_pants
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'away_cap'] = most_common_cap
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'away_jersey_code'] = most_common_code
            
            print(f"  ✓ gamePk {gamepk} ({team_name}): {most_common_jersey}")

print(f"\nFinal missing count:")
print(f"  Home jerseys: {df_uniforms['home_jersey'].isna().sum()}")
print(f"  Away jerseys: {df_uniforms['away_jersey'].isna().sum()}")



Filling 2 missing home jerseys with most common jersey for each team...
  ✓ gamePk 776823 (Los Angeles Dodgers): Dodgers Home White Jersey
  ✓ gamePk 776714 (Boston Red Sox): Red Sox Home White Jersey ("Red Sox")

Filling 1 missing away jerseys with most common jersey for each team...
  ✓ gamePk 776823 (Toronto Blue Jays): Blue Jays Alt All Blue Jersey

Final missing count:
  Home jerseys: 0
  Away jerseys: 0


In [37]:
# Final summary of uniform data completeness
print("=" * 50)
print("UNIFORM DATA SUMMARY")
print("=" * 50)
print(f"\nTotal games: {len(df_uniforms)}")
print(f"Games with home jersey: {df_uniforms['home_jersey'].notna().sum()}")
print(f"Games with away jersey: {df_uniforms['away_jersey'].notna().sum()}")
print(f"Games with complete uniform data (both jerseys): {(df_uniforms['home_jersey'].notna() & df_uniforms['away_jersey'].notna()).sum()}")
print(f"\nHome jersey completion rate: {df_uniforms['home_jersey'].notna().sum() / len(df_uniforms) * 100:.2f}%")
print(f"Away jersey completion rate: {df_uniforms['away_jersey'].notna().sum() / len(df_uniforms) * 100:.2f}%")


UNIFORM DATA SUMMARY

Total games: 2430
Games with home jersey: 2430
Games with away jersey: 2430
Games with complete uniform data (both jerseys): 2430

Home jersey completion rate: 100.00%
Away jersey completion rate: 100.00%


In [40]:
# Fill missing caps based on the jersey code (most common cap for that jersey)
# Fill missing home caps
missing_home_caps = df_uniforms[df_uniforms['home_cap'].isna() & df_uniforms['home_jersey_code'].notna()]
if len(missing_home_caps) > 0:
    print(f"\nFilling {len(missing_home_caps)} missing home caps using jersey code...")
    
    for idx, row in missing_home_caps.iterrows():
        jersey_code = row['home_jersey_code']
        gamepk = row['gamePk']
        
        # Find most common cap for this jersey code
        matching_outfits = df_uniforms[
            (df_uniforms['home_jersey_code'] == jersey_code) & 
            (df_uniforms['home_cap'].notna())
        ]
        
        if len(matching_outfits) > 0:
            most_common_cap = matching_outfits['home_cap'].mode()[0]
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'home_cap'] = most_common_cap
            print(f"  ✓ gamePk {gamepk} ({jersey_code}): {most_common_cap}")
        else:
            print(f"  ⚠️ gamePk {gamepk}: No cap data found for jersey code {jersey_code}")

# Fill missing away caps
missing_away_caps = df_uniforms[df_uniforms['away_cap'].isna() & df_uniforms['away_jersey_code'].notna()]
if len(missing_away_caps) > 0:
    print(f"\nFilling {len(missing_away_caps)} missing away caps using jersey code...")
    
    for idx, row in missing_away_caps.iterrows():
        jersey_code = row['away_jersey_code']
        gamepk = row['gamePk']
        
        # Find most common cap for this jersey code
        matching_outfits = df_uniforms[
            (df_uniforms['away_jersey_code'] == jersey_code) & 
            (df_uniforms['away_cap'].notna())
        ]
        
        if len(matching_outfits) > 0:
            most_common_cap = matching_outfits['away_cap'].mode()[0]
            df_uniforms.loc[df_uniforms['gamePk'] == gamepk, 'away_cap'] = most_common_cap
            print(f"  ✓ gamePk {gamepk} ({jersey_code}): {most_common_cap}")
        else:
            print(f"  ⚠️ gamePk {gamepk}: No cap data found for jersey code {jersey_code}")

print(f"\nFinal missing cap count:")
print(f"  Home caps: {df_uniforms['home_cap'].isna().sum()}")
print(f"  Away caps: {df_uniforms['away_cap'].isna().sum()}")



Filling 5 missing home caps using jersey code...
  ✓ gamePk 778415 (134_jersey_1_2025): Pirates Primary Black Hat
  ✓ gamePk 777304 (118_jersey_3_2025): Royals Primary Blue Hat
  ✓ gamePk 777244 (133_jersey_1_2025): Athletics Primary Green Top Yellow Bill Hat
  ✓ gamePk 776879 (134_jersey_4_2025): Pirates Primary Black Hat
  ✓ gamePk 776368 (147_jersey_1_2025): Yankees Primary Navy Hat

Filling 5 missing away caps using jersey code...
  ✓ gamePk 777891 (133_jersey_2_2025): Athletics Alt All Green Hat, Yellow Trim "A's"
  ✓ gamePk 777879 (133_jersey_2_2025): Athletics Alt All Green Hat, Yellow Trim "A's"
  ✓ gamePk 777867 (133_jersey_2_2025): Athletics Alt All Green Hat, Yellow Trim "A's"
  ✓ gamePk 777304 (119_jersey_3_2025): Dodgers Primary Blue Hat
  ✓ gamePk 776371 (121_jersey_2_2025): Mets Primary Blue Hat

Final missing cap count:
  Home caps: 0
  Away caps: 0


In [41]:
df_uniforms.head()

,gamePk,home_jersey,home_pants,home_cap,home_jersey_code,away_jersey,away_pants,away_cap,away_jersey_code
0,778563,Cubs Home Pinstripe Jersey,Cubs Home Pinstripe Pants,Cubs Primary Blue Hat,112_jersey_1_2025,"Dodgers Alt Road Grey ""Dodgers"" Jersey",Dodgers Road Grey Pants,Dodgers Primary Blue Hat,119_jersey_3_2025
1,778564,Cubs Away Grey Jersey,Cubs Away Grey Pants,Cubs Primary Blue Hat,112_jersey_2_2025,Dodgers Home White Jersey,Dodgers Home White Pants,Dodgers Primary Blue Hat,119_jersey_1_2025
2,778557,Yankees Home Pinstripe Jersey,Yankees Home Pinstripe Pants,Yankees Primary Navy Hat,147_jersey_1_2025,Brewers Alt Blue Jersey,Brewers Road Grey Pants,Brewers Alt Yellow Front Hat,158_jersey_4_2025
3,778556,Blue Jays Home White Jersey,Blue Jays Home White Pants,Blue Jays Primary All Blue Hat,141_jersey_1_2025,Orioles Alt Black Jersey,Orioles Away Grey Pants,"Orioles Alt Black Front Orange Bill ""O's"" Hat",110_jersey_3_2025
4,778545,Padres Home White Pinstripe Jersey,Padres Home White Pinstripe Pants,Padres Primary All Brown Hat,135_jersey_1_2025,Braves Road Grey Jersey,Braves Road Grey Pants,Braves Alt All Navy Hat,144_jersey_2_2025


In [42]:
df_uniforms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2430 entries, 0 to 2429
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   gamePk            2430 non-null   int64 
 1   home_jersey       2430 non-null   object
 2   home_pants        2430 non-null   object
 3   home_cap          2430 non-null   object
 4   home_jersey_code  2430 non-null   object
 5   away_jersey       2430 non-null   object
 6   away_pants        2430 non-null   object
 7   away_cap          2430 non-null   object
 8   away_jersey_code  2430 non-null   object
dtypes: int64(1), object(8)
memory usage: 171.0+ KB


In [43]:
# Save the three individual dataframes before merging
print("Saving individual dataframes to CSV...")

# Save games data
df_games.to_csv('mlb_2025_games.csv', index=False)
print(f"✓ Saved df_games: {len(df_games)} rows → mlb_2025_games.csv")

# Save attendance data
df_attendance.to_csv('mlb_2025_attendance.csv', index=False)
print(f"✓ Saved df_attendance: {len(df_attendance)} rows → mlb_2025_attendance.csv")

# Save uniform data
df_uniforms.to_csv('mlb_2025_uniforms.csv', index=False)
print(f"✓ Saved df_uniforms: {len(df_uniforms)} rows → mlb_2025_uniforms.csv")

print("\nAll individual dataframes saved successfully!")


Saving individual dataframes to CSV...
✓ Saved df_games: 2430 rows → mlb_2025_games.csv
✓ Saved df_attendance: 2430 rows → mlb_2025_attendance.csv
✓ Saved df_uniforms: 2430 rows → mlb_2025_uniforms.csv

All individual dataframes saved successfully!


## Step 5: Combine All Data into Single DataFrame


In [44]:
# Merge all dataframes on gamePk
df_final = df_games.merge(df_attendance, on='gamePk', how='left')
df_final = df_final.merge(df_uniforms, on='gamePk', how='left')

print(f"Final dataset shape: {df_final.shape}")
print(f"\nColumns: {list(df_final.columns)}")
print(f"\nFirst few rows:")
df_final.head(10)


Final dataset shape: (2430, 42)

Columns: ['gamePk', 'date', 'gameDate', 'season', 'gameType', 'status', 'statusCode', 'home_team_id', 'home_team_name', 'home_score', 'home_wins', 'home_losses', 'home_pct', 'home_is_winner', 'away_team_id', 'away_team_name', 'away_score', 'away_wins', 'away_losses', 'away_pct', 'away_is_winner', 'venue_id', 'venue_name', 'doubleHeader', 'gameNumber', 'dayNight', 'description', 'scheduledInnings', 'seriesGameNumber', 'gamesInSeries', 'attendance', 'attendance_high_for_date', 'attendance_low_for_date', 'is_doubleheader_date', 'home_jersey', 'home_pants', 'home_cap', 'home_jersey_code', 'away_jersey', 'away_pants', 'away_cap', 'away_jersey_code']

First few rows:


,gamePk,date,gameDate,season,gameType,status,statusCode,home_team_id,home_team_name,home_score,...,attendance_low_for_date,is_doubleheader_date,home_jersey,home_pants,home_cap,home_jersey_code,away_jersey,away_pants,away_cap,away_jersey_code
0,778563,2025-03-18,2025-03-18T10:10:00Z,2025,R,Final,F,112,Chicago Cubs,1.0,...,42365.0,False,Cubs Home Pinstripe Jersey,Cubs Home Pinstripe Pants,Cubs Primary Blue Hat,112_jersey_1_2025,"Dodgers Alt Road Grey ""Dodgers"" Jersey",Dodgers Road Grey Pants,Dodgers Primary Blue Hat,119_jersey_3_2025
1,778564,2025-03-19,2025-03-19T10:10:00Z,2025,R,Final,F,112,Chicago Cubs,3.0,...,42367.0,False,Cubs Away Grey Jersey,Cubs Away Grey Pants,Cubs Primary Blue Hat,112_jersey_2_2025,Dodgers Home White Jersey,Dodgers Home White Pants,Dodgers Primary Blue Hat,119_jersey_1_2025
2,778557,2025-03-27,2025-03-27T19:05:00Z,2025,R,Final,F,147,New York Yankees,4.0,...,46208.0,False,Yankees Home Pinstripe Jersey,Yankees Home Pinstripe Pants,Yankees Primary Navy Hat,147_jersey_1_2025,Brewers Alt Blue Jersey,Brewers Road Grey Pants,Brewers Alt Yellow Front Hat,158_jersey_4_2025
3,778556,2025-03-27,2025-03-27T19:07:00Z,2025,R,Final,F,141,Toronto Blue Jays,2.0,...,40734.0,False,Blue Jays Home White Jersey,Blue Jays Home White Pants,Blue Jays Primary All Blue Hat,141_jersey_1_2025,Orioles Alt Black Jersey,Orioles Away Grey Pants,"Orioles Alt Black Front Orange Bill ""O's"" Hat",110_jersey_3_2025
4,778545,2025-03-27,2025-03-27T20:10:00Z,2025,R,Final,F,135,San Diego Padres,7.0,...,45568.0,False,Padres Home White Pinstripe Jersey,Padres Home White Pinstripe Pants,Padres Primary All Brown Hat,135_jersey_1_2025,Braves Road Grey Jersey,Braves Road Grey Pants,Braves Alt All Navy Hat,144_jersey_2_2025
5,778553,2025-03-27,2025-03-27T20:05:00Z,2025,R,Final,F,140,Texas Rangers,2.0,...,37585.0,False,Rangers Home White Jersey,Rangers Home White Pants,Rangers Primary All Blue Hat,140_jersey_1_2025,Red Sox Away Grey Jersey,Red Sox Away Grey Pants,"Red Sox Primary Blue ""B"" Hat",111_jersey_2_2025
6,778555,2025-03-27,2025-03-27T20:05:00Z,2025,R,Final,F,120,Washington Nationals,3.0,...,41231.0,False,Nationals Home White Jersey,Nationals Home White Pants,"Nationals Primary Red ""W"" Hat",120_jersey_1_2025,Phillies Away Grey Jersey,Phillies Away Grey Pants,Phillies Primary Red Hat,143_jersey_2_2025
7,778558,2025-03-27,2025-03-27T20:10:00Z,2025,R,Final,F,118,Kansas City Royals,4.0,...,39393.0,False,Royals Alt Baby Blue Jersey,Royals Alt Baby Blue Pants,Royals Alt White Front R-Crown Hat,118_jersey_3_2025,Guardians Away Grey Jersey,Guardians Away Grey Pants,"Guardians Blue Top, Red Bill ""C"" Hat",114_jersey_2_2025
8,778561,2025-03-27,2025-03-27T20:10:00Z,2025,R,Final,F,113,Cincinnati Reds,4.0,...,43876.0,False,Reds Home White Jersey,Reds Home White Pants,"Reds Primary All Red ""C"" Hat",113_jersey_1_2025,Giants Away Grey Jersey,Giants Away Grey Pants,Giants Primary All Black Hat,137_jersey_2_2025
9,778559,2025-03-27,2025-03-27T20:10:00Z,2025,R,Final,F,117,Houston Astros,3.0,...,42305.0,False,Astros Home White Jersey,Astros Home White Pants,Astros Primary All Blue Hat,117_jersey_1_2025,Mets Away Grey Jersey,Mets Away Grey Pants Blue/Orange Stripes,Mets Primary Blue Hat,121_jersey_2_2025


## Step 6: Data Quality Check


In [45]:
# Check data completeness
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

print(f"\nTotal games: {len(df_final)}")
print(f"\nGames by status:")
print(df_final['status'].value_counts())

print(f"\nAttendance data coverage:")
print(f"  - Games with attendance: {df_final['attendance'].notna().sum()} ({df_final['attendance'].notna().sum()/len(df_final)*100:.1f}%)")
print(f"  - Games without attendance: {df_final['attendance'].isna().sum()}")

print(f"\nUniform data coverage:")
print(f"  - Games with home jersey: {df_final['home_jersey'].notna().sum()} ({df_final['home_jersey'].notna().sum()/len(df_final)*100:.1f}%)")
print(f"  - Games with away jersey: {df_final['away_jersey'].notna().sum()} ({df_final['away_jersey'].notna().sum()/len(df_final)*100:.1f}%)")

print(f"\nDouble headers: {(df_final['doubleHeader'] == 'Y').sum()}")

print(f"\nDate range: {df_final['date'].min()} to {df_final['date'].max()}")


DATA QUALITY REPORT

Total games: 2430

Games by status:
status
Final              2425
Completed Early       5
Name: count, dtype: int64

Attendance data coverage:
  - Games with attendance: 2430 (100.0%)
  - Games without attendance: 0

Uniform data coverage:
  - Games with home jersey: 2430 (100.0%)
  - Games with away jersey: 2430 (100.0%)

Double headers: 12

Date range: 2025-03-18 to 2025-09-28


## Step 7: Sample Analysis


In [46]:
# Show some interesting subsets
print("Sample games with complete data:")
complete_data = df_final[
    df_final['attendance'].notna() & 
    df_final['home_jersey'].notna() & 
    df_final['away_jersey'].notna()
]

if len(complete_data) > 0:
    print(f"\nGames with complete data: {len(complete_data)}")
    display(complete_data[[
        'date', 'home_team_name', 'away_team_name', 
        'home_score', 'away_score', 'attendance', 
        'home_jersey', 'away_jersey', 'venue_name'
    ]].head(10))
else:
    print("No games with complete data yet (games may not have been played)")


Sample games with complete data:

Games with complete data: 2430


,date,home_team_name,away_team_name,home_score,away_score,attendance,home_jersey,away_jersey,venue_name
0,2025-03-18,Chicago Cubs,Los Angeles Dodgers,1.0,4.0,42365.0,Cubs Home Pinstripe Jersey,"Dodgers Alt Road Grey ""Dodgers"" Jersey",Tokyo Dome
1,2025-03-19,Chicago Cubs,Los Angeles Dodgers,3.0,6.0,42367.0,Cubs Away Grey Jersey,Dodgers Home White Jersey,Tokyo Dome
2,2025-03-27,New York Yankees,Milwaukee Brewers,4.0,2.0,46208.0,Yankees Home Pinstripe Jersey,Brewers Alt Blue Jersey,Yankee Stadium
3,2025-03-27,Toronto Blue Jays,Baltimore Orioles,2.0,12.0,40734.0,Blue Jays Home White Jersey,Orioles Alt Black Jersey,Rogers Centre
4,2025-03-27,San Diego Padres,Atlanta Braves,7.0,4.0,45568.0,Padres Home White Pinstripe Jersey,Braves Road Grey Jersey,Petco Park
5,2025-03-27,Texas Rangers,Boston Red Sox,2.0,5.0,37585.0,Rangers Home White Jersey,Red Sox Away Grey Jersey,Globe Life Field
6,2025-03-27,Washington Nationals,Philadelphia Phillies,3.0,7.0,41231.0,Nationals Home White Jersey,Phillies Away Grey Jersey,Nationals Park
7,2025-03-27,Kansas City Royals,Cleveland Guardians,4.0,7.0,39393.0,Royals Alt Baby Blue Jersey,Guardians Away Grey Jersey,Kauffman Stadium
8,2025-03-27,Cincinnati Reds,San Francisco Giants,4.0,6.0,43876.0,Reds Home White Jersey,Giants Away Grey Jersey,Great American Ball Park
9,2025-03-27,Houston Astros,New York Mets,3.0,1.0,42305.0,Astros Home White Jersey,Mets Away Grey Jersey,Daikin Park


In [47]:
# Attendance statistics for completed games
completed_games = df_final[df_final['status'] == 'Final']
if len(completed_games) > 0 and completed_games['attendance'].notna().sum() > 0:
    print("\nAttendance Statistics for Completed Games:")
    print(f"  - Average attendance: {completed_games['attendance'].mean():,.0f}")
    print(f"  - Median attendance: {completed_games['attendance'].median():,.0f}")
    print(f"  - Min attendance: {completed_games['attendance'].min():,.0f}")
    print(f"  - Max attendance: {completed_games['attendance'].max():,.0f}")
    print(f"  - Total attendance: {completed_games['attendance'].sum():,.0f}")



Attendance Statistics for Completed Games:
  - Average attendance: 29,473
  - Median attendance: 30,540
  - Min attendance: 2,518
  - Max attendance: 91,032
  - Total attendance: 71,472,744


In [48]:
# Sample uniform variety
games_with_uniforms = df_final[df_final['home_jersey'].notna()]
if len(games_with_uniforms) > 0:
    print("\nSample of Uniform Variety:")
    print(f"\nUnique home jerseys: {df_final['home_jersey'].nunique()}")
    print(f"\nSample jerseys worn:")
    print(df_final['home_jersey'].value_counts().head(10))



Sample of Uniform Variety:

Unique home jerseys: 105

Sample jerseys worn:
home_jersey
Yankees Home Pinstripe Jersey         81
Tigers Home White Jersey              70
Dodgers Home White Jersey             69
White Sox Home Pinstripe Jersey       68
Cubs Home Pinstripe Jersey            66
Angels Home White Jersey              65
Rays Home White Jersey ("Rays")       60
Athletics Home White Jersey           59
Cardinals Home White Jersey           56
Padres Home White Pinstripe Jersey    55
Name: count, dtype: int64


## Step 8: Save Dataset


In [49]:
# Save to CSV
output_file = 'mlb_2025_complete_dataset.csv'
df_final.to_csv(output_file, index=False)
print(f"Dataset saved to {output_file}")

# Also save to parquet for better performance with large datasets
output_parquet = 'mlb_2025_complete_dataset.parquet'
df_final.to_parquet(output_parquet, index=False)
print(f"Dataset also saved to {output_parquet}")


Dataset saved to mlb_2025_complete_dataset.csv
Dataset also saved to mlb_2025_complete_dataset.parquet


In [50]:
# Display final summary
print("\n" + "=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)
print(f"\nTotal records: {len(df_final)}")
print(f"Total columns: {len(df_final.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")



FINAL DATASET SUMMARY

Total records: 2430
Total columns: 42

Column list:
   1. gamePk
   2. date
   3. gameDate
   4. season
   5. gameType
   6. status
   7. statusCode
   8. home_team_id
   9. home_team_name
  10. home_score
  11. home_wins
  12. home_losses
  13. home_pct
  14. home_is_winner
  15. away_team_id
  16. away_team_name
  17. away_score
  18. away_wins
  19. away_losses
  20. away_pct
  21. away_is_winner
  22. venue_id
  23. venue_name
  24. doubleHeader
  25. gameNumber
  26. dayNight
  27. description
  28. scheduledInnings
  29. seriesGameNumber
  30. gamesInSeries
  31. attendance
  32. attendance_high_for_date
  33. attendance_low_for_date
  34. is_doubleheader_date
  35. home_jersey
  36. home_pants
  37. home_cap
  38. home_jersey_code
  39. away_jersey
  40. away_pants
  41. away_cap
  42. away_jersey_code
